## Cargar los datos ##

In [64]:
import numpy as np
import pandas as pd

In [65]:
df = pd.read_csv('data/GenreAll.csv', skipinitialspace=True)

df = df.dropna(subset=['genre']) # eliminar filas que no tienen género

#feature que queremos predecir
y = df['genre'] 

#features para el entrenamiento
X = df.drop(columns=["genre"]).copy()

# Convertir columnas binarias de texto en 0 y 1
for col in X.select_dtypes(include=["object"]).columns:
    values = X[col].dropna().unique()

    if len(values) == 2:
        X[col] = X[col].map({
            values[0]: 0,
            values[1]: 1
        })
#asegurar que solo hay valores numéricos
X = X.select_dtypes(include=np.number)

print("Número de canciones:", len(df))
print("Número de features:", X.shape[1])
print("\nCanciones por género:")
print(y.value_counts())

df.head()

Número de canciones: 993
Número de features: 240

Canciones por género:
genre
blu    100
cou    100
dis    100
hip    100
jaz    100
met    100
pop    100
reg    100
roc    100
cla     93
Name: count, dtype: int64


/tmp/ipykernel_8567/3862506610.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include=["object"]).columns:


,highlevel_danceability_value,highlevel_equalization_profile_value,highlevel_excitement_value,highlevel_intensity_value,highlevel_speech_music_value,highlevel_voice_instrumental_value,loudness_dynamic_complexity_dvar,loudness_dynamic_complexity_mean,loudness_dynamic_complexity_var,loudness_larm_dvar,...,tonal_chords_strength_dvar,tonal_chords_strength_mean,tonal_chords_strength_var,tonal_dissonance_dvar,tonal_dissonance_mean,tonal_dissonance_var,tonal_key_mode_value,tonal_key_strength_value,tonal_tuning_equal_tempered_deviation_value,genre
0,0.477744,0.372766,0.029087,0.093526,speech,instrumental,-0.366526,0.832351,-0.398487,0.262413,...,0.175405,0.101673,0.989475,0.167484,-0.091051,0.085449,major,0.162259,2.423830,blu
1,0.271161,0.372766,-1.283803,-0.615420,speech,voice,0.275184,1.901008,-0.287263,0.078360,...,-0.157583,-0.652126,-0.992276,0.606661,-0.468630,0.416914,major,0.230571,-0.207952,blu
2,0.089011,0.372766,0.029087,1.511419,music,instrumental,-0.177390,0.046932,0.035041,-0.157974,...,-0.001059,-0.220255,0.978014,-0.236338,0.470580,-0.205887,major,1.024735,-0.211034,blu
3,0.243503,0.372766,0.029087,-0.615420,music,instrumental,-0.319420,-0.953075,0.239607,-0.659566,...,0.214463,0.853728,1.327351,-0.542341,0.642223,-0.791110,major,0.079033,0.148357,blu
4,0.594883,-1.940715,-1.283803,0.093526,speech,instrumental,-0.224956,1.154507,-0.420087,0.281666,...,-0.140930,-0.080519,-1.163700,0.094741,-0.164770,-0.277919,major,-0.337966,-0.277459,blu


## Definición de los modelos ##

Un pipeline es una cadena de pasos que se ejecutan en orden. Por ejemplo, antes de entrenar un modelo, podemos necesitar rellenar datos que faltan y normalizar las características.

`make_pipeline` permite crear un pipeline combinando varios pasos

`SimpleImputer` sirve para rellenar los valores que faltan en los datos, por ejemplo, cuando una característica musical contiene NaN. se puede usar `StandardScaler` para rellenar esos datos vacíos

In [66]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

#modelos
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

In [67]:
models = {
    "Logistic Regression": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000))]),
        {"model__C": [0.1, 1, 10]}),

    "k-NN": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier())]),
        {
            "model__n_neighbors": [3, 5, 7],
            "model__weights": ["uniform", "distance"]
        }
    ),

    "Decision Tree": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DecisionTreeClassifier(random_state=42))]),
        {
            "model__max_depth": [5, 10, None],
            "model__min_samples_leaf": [1, 3]
        }
    ),

    "Artificial Neural Network": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                max_iter=500,
                random_state=42))]),
        {
            "model__hidden_layer_sizes": [(50,), (100,), (100, 50)],
            "model__alpha": [0.0001, 0.01]
        }
    )
}
# Identificar las columnas numéricas/no numéricas
numeric_columns = X.select_dtypes(include=np.number).columns
categorical_columns = X.select_dtypes(exclude=np.number).columns


## Definir la validación cruzada
 
Mismo esquema de 10 folds para comparar todos los algoritmos

In [68]:
from sklearn.model_selection import StratifiedKFold

In [69]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42)

## Entrenar y evaluar los modelos

Esta es la celda principal. `GridSearchCV` probará todas las configuraciones usando validación cruzada de 10 folds y seleccionará la que obtenga la mejor accuracy media para cada algoritmo

In [70]:
from sklearn.model_selection import GridSearchCV

In [71]:

results = []
best_models = {}

for name, (pipeline, params) in models.items():

    print("\nEvaluando:", name)

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring="accuracy",
        cv=cv,
        return_train_score=True,
        n_jobs=-1
    )

    search.fit(X, y)

    best_models[name] = search

    # Guardar los resultados de todas las configuraciones
    for i in range(len(search.cv_results_["params"])):
        results.append({
            "Algorithm": name,
            "Parameters": search.cv_results_["params"][i],
            "Train accuracy": search.cv_results_["mean_train_score"][i],
            "CV accuracy": search.cv_results_["mean_test_score"][i],
            "CV standard deviation": search.cv_results_["std_test_score"][i]
        })

    print("Mejores parámetros:", search.best_params_)
    print("Best 10-fold CV accuracy:", round(search.best_score_, 4))

results_df = pd.DataFrame(results)

print("\nTodos los resultados:")
display(
    results_df.sort_values(
        "CV accuracy",
        ascending=False
    ).reset_index(drop=True)
)




Evaluando: Logistic Regression


Mejores parámetros: {'model__C': 0.1}
Best 10-fold CV accuracy: 0.8408

Evaluando: k-NN
Mejores parámetros: {'model__n_neighbors': 7, 'model__weights': 'distance'}
Best 10-fold CV accuracy: 0.7471

Evaluando: Decision Tree
Mejores parámetros: {'model__max_depth': None, 'model__min_samples_leaf': 1}
Best 10-fold CV accuracy: 0.6353

Evaluando: Artificial Neural Network
Mejores parámetros: {'model__alpha': 0.01, 'model__hidden_layer_sizes': (50,)}
Best 10-fold CV accuracy: 0.8429

Todos los resultados:


,Algorithm,Parameters,Train accuracy,CV accuracy,CV standard deviation
0,Artificial Neural Network,"{'model__alpha': 0.01, 'model__hidden_layer_si...",0.999105,0.842869,0.042383
1,Artificial Neural Network,"{'model__alpha': 0.0001, 'model__hidden_layer_...",0.999105,0.841859,0.043779
2,Logistic Regression,{'model__C': 0.1},0.984558,0.840848,0.043618
3,Artificial Neural Network,"{'model__alpha': 0.01, 'model__hidden_layer_si...",0.999105,0.837818,0.036397
4,Artificial Neural Network,"{'model__alpha': 0.0001, 'model__hidden_layer_...",0.999105,0.837808,0.039117
5,Logistic Regression,{'model__C': 1},0.998210,0.834798,0.046354
6,Logistic Regression,{'model__C': 10},0.999105,0.831808,0.042235
7,Artificial Neural Network,"{'model__alpha': 0.01, 'model__hidden_layer_si...",0.999105,0.827697,0.037183
8,Artificial Neural Network,"{'model__alpha': 0.0001, 'model__hidden_layer_...",0.999105,0.825687,0.036235
9,k-NN,"{'model__n_neighbors': 7, 'model__weights': 'd...",0.999105,0.747111,0.052460
